# Create predictand dataset

To plug into fully interacted ("Agespec_interaction") Carleton et al. model: 
- Long-run temperature: From tassumpoly, degree 1, / 365
- Population: From input dataset + SSP outputs
- GDP PC: From input dataset + SSP outputs

(the SSP outputs start at 2010, which may be too late for GWL1, and definitely 0.61 - so we take population and GDP pc from the mortality data dataset)

In [13]:
import xarray as xr
import numpy as np
import pandas as pd
import re
import glob

from funcs_support import get_params, get_filepaths, utility_save
dir_list = get_params()

In [2]:
from distributed import Client
# Start dask client
client = Client()
display(client)

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33593 instead
  warnings.warn(


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/33593/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/33593/status,Workers: 6
Total threads: 18,Total memory: 72.00 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:34323,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/33593/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33373,Total threads: 3
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/schwarzwald/climate/proxy/33841/status,Memory: 12.00 GiB
Nanny: tcp://127.0.0.1:41035,


In [18]:
df = get_filepaths(source_dir = 'proc')
df = df.query('varname == "tassumpoly"')

In [20]:
import geopandas as gpd
# Load Carleton et al. "Impact Regions"
gdf = gpd.read_file(dir_list['aux']+'geo_data/impact-region.shp')

# Remove antarctica (why is it on there...)
gdf = gdf.loc[gdf.hierid != "ATA",:]

/glade/u/home/schwarzwald/.conda/envs/enso_imp/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: /glade/u/home/schwarzwald/projects/hle_iv/aux_data/geo_data/impact-region.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(


In [25]:
gdf

,gadmid,hierid,color,ISO,AREA,PERIMETER,geometry
0,28115,CAN.1.2.28,1,CAN,0.001442,0.184641,"POLYGON ((-110.05459 53.3073, -110.00587 53.30..."
1,28116,CAN.1.17.403,2,CAN,0.000081,0.041724,"POLYGON ((-111.23055 52.91943, -111.22076 52.9..."
2,28119,CAN.2.34.951,3,CAN,0.000490,0.091736,"POLYGON ((-127.68527 55.2957, -127.69742 55.32..."
3,28120,CAN.11.259.4274,4,CAN,0.001212,0.183588,"POLYGON ((-77.7308 55.31879, -77.7103 55.3176,..."
4,28124,CAN.11.269.4448,5,CAN,0.489323,6.556768,"POLYGON ((-66.2594 54.99975, -65.24999 55, -64..."
...,...,...,...,...,...,...,...
24373,6905,BWA.1,24374,BWA,12.798809,18.094620,"POLYGON ((26.01723 -23.55865, 25.86139 -23.355..."
24374,6902,BWA.7,24375,BWA,11.378698,16.661345,"POLYGON ((22.50413 -18.11382, 22.98841 -18.018..."
24375,6915,BWA.6.16,24376,BWA,0.459842,3.292925,"POLYGON ((27.28497 -20.49781, 27.35821 -20.473..."
24376,6909,BWA.5,24377,BWA,3.274455,8.503536,"POLYGON ((23.05667 -23.31139, 23.55222 -23.313..."


In [ ]:
gdf = gdf.dissolve

# Load or generate weightmap between 0.25 grid and impact regions
wm_fn = dir_list['aux']+'wm_0-25deg_adm1_impactregions'

# Get sample ds for weightmap
ds = xr.open_zarr(df.iloc[0]['path'])

if not os.path.exists(wm_fn):
    wms['impactregions'] = xa.pixel_overlaps(ds,gdfs['impactregions'])
    wms['impactregions'].to_file(wm_fn)
else:
    wms['impactregions'] = xa.read_wm(wm_fn)

In [ ]:
# Make ADM1 and ADM0 level versions of the polygons
gdf_sub['all'].dissolve(by=['iso','adm1_id']).reset_index().drop('adm2_id',axis=1)
gdf_sub['adm0'] = gdf_sub['all'].dissolve('adm0').reset_index().drop(['adm1_id','adm2_id'],axis=1)


,varname,freq,model,exp,run,gwl,proj_method,proj_base,filetype,dwnscl_method,dwnscl_target,suffix,path,time,unlabeled,mtime
9,tassumpoly,day,EC-Earth3-Veg,ssp585,r4i1p1f1,ALLGWLs,QDM,JRA-3Q,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 13:29:48+00:00
19,tassumpoly,day,EC-Earth3-Veg,ssp245,r5i1p1f1,ALLGWLs,QDM,ERA5,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 13:30:17+00:00
22,tassumpoly,day,EC-Earth3-Veg,ssp585,r1i1p1f1,ALLGWLs,QDM,JRA-3Q,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 13:30:08+00:00
29,tassumpoly,day,EC-Earth3-Veg,ssp370,r2i1p1f1,ALLGWLs,QDM,JRA-3Q,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-10-11 03:22:51+00:00
40,tassumpoly,day,EC-Earth3-Veg,ssp585,r4i1p1f1,ALLGWLs,QDM,ERA5,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 13:28:05+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,tassumpoly,day,FGOALS-g3,ssp585,r3i1p1f1,ALLGWLs,QDM,ERA5-025,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 16:31:47+00:00
252,tassumpoly,day,FGOALS-g3,ssp370,r4i1p1f1,ALLGWLs,QDM,JRA-3Q,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-10-11 05:40:19+00:00
261,tassumpoly,day,FGOALS-g3,ssp585,r4i1p1f1,ALLGWLs,QDM,GMFD,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-09-03 16:30:55+00:00
272,tassumpoly,day,FGOALS-g3,ssp370,r3i1p1f1,ALLGWLs,QDM,JRA-3Q,zarr,QPLAD,025deg,NaN,/glade/derecho/scratch/schwarzwald/climate_pro...,NaN,NaN,2025-10-11 05:18:34+00:00


In [3]:
#------------------------------------
# 1. Load precalculated, aggregated 
# tas sum polys
#------------------------------------
dir_list['aux_bigmem'] = '/glade/derecho/scratch/schwarzwald/aux_data/'

fn_clim = dir_list['aux_bigmem']+'tassumpoly_agg_impactregions_ALLFILES.zarr'
    
ds_clim = xr.open_zarr(fn_clim)
    
# Drop extra metadata, for convenience
ds_clim = ds_clim.drop_vars(['ISO','AREA','PERIMETER','color','gadmid'])

# Get average 20-year temperature as 1/365 of the average sum of temperature
ds_clim = ds_clim.tassumpoly.sel(degree = 1) / 365

# Multiindex components aren't loaded into memory by default, but are 
# needed for subsetting (this doesn't load data into memory, just the
# coordinates that arent dimensions but subunits of the MultiIndex)
for k in ds_clim.coords:
    ds_clim[k] = ds_clim[k].load()

In [4]:
#------------------------------------
# 2. Load gwls for each GCM run
#------------------------------------
gwls = xr.open_dataset(dir_list['aux']+'gwl_ann_CMIP6_ALLEXPs_ALLRUNs_1860-2090_wsomeESGFruns.nc')
gwls = gwls.tasanom.groupby(['experiment','year']).mean()

gwls = gwls.dropna('experiment',how='all')

gwl_years = gwls.year[np.abs(gwls-ds_clim.gwl).argmin('year')].drop_vars('year')
# argmin doesn't put nans where there is no data (i.e., where an experiment
# has no data at a given GWL 
gwl_years = gwl_years.where(ds_clim.gwl <= gwls.max('year'))
# Worth considering whether to just use the subset of model / runs from ds_clim
# (well, hm. The thought behind this is I just pick the SSP3 to match up with 
# Carleton, with the idea being that the GCMs tell us what's happening at a GWL, 
# but that the year the GWL is passed is taken from the SSP)

In [5]:
#------------------------------------
# 3. Load GDP and population from SSPs
#------------------------------------

ssp_pred_fns = glob.glob(dir_list['aux']+'SSP*.nc4')
# Convert to the full SSP codes in the gwls file (since SSP1 can be
# SSP119 or SSP126 for example)
preds = [(
          xr.open_dataset(fn).
                  expand_dims({'experiment':[exp for exp in gwls.experiment.values 
                                      if re.search(r'^'+re.split(r'\.',re.split(r'\/',fn)[-1])[0].lower(),
                                                    exp)]
                      }
                     )
         ) 
         for fn in ssp_pred_fns]
preds = xr.concat(preds,dim='experiment')


# Set stats from before the first pred year to the
# stats in the first pred year
preds = xr.concat([preds.isel(year=0,drop=True).expand_dims({'year':np.arange(gwl_years.min().values,
                                                           preds.year.min().values).astype(int)}),
          preds],
          dim='year')

In [6]:
#------------------------------------
# 4. Get GDP and population for the 
# average year in an SSP for a given 
# GWL
#------------------------------------
# Get pop stats for the years needed for each gwl
preds = preds.sel(year=gwl_years.sel(experiment=preds.experiment))

In [7]:
#------------------------------------
# 5. Add long-run climate averages
#------------------------------------

# Rename experiment in preds to differentiate from the `idv`
# experiment in the climate data 
preds = preds.rename({'experiment':'pred_experiment',
                      'region':'hierid',
                      'model':'pred_model'})

preds.pred_experiment.attrs['long_name'] = 'SSP used at a given GWL for GDP / population'

# Add long-run climate average 
preds['lr_tavg'] = ds_clim

In [10]:
#------------------------------------
# 6. Save
#------------------------------------
# Housekeeping
preds.attrs['DESCRIPTION'] = ('Population, per-capita GDP, and long-run climate variables '+
                              'for a given GWL, set by a given `pred_experiment`. i.e., '+
                              'for `pred_experiment == ssp370`, `gwl = 2`, population and per-capita GDP '+
                              'are taken from the year of SSP3 in which the average CMIP6 SSP3.70 '+
                              'run crosses 2 degrees of GMST vs. its 1850-1900 average, and the long-run '+
                              'climate is the 20-year mean local temperature over the years around that '+
                              'model-experiment-run reaching 2 degrees of GWL. Note that the experiment in '+
                              'idv is not necessarily the same as the experiment in `pred_experiment`; i.e., '+
                              'the climate variables are by GWL, regardless of pathway or SSP taken to get there '+
                              ', while the population / GDP data is by the SSP in `pred_experiment`.')
preds.attrs['SOURCE'] = 'create_pred_dataset.ipynb'

# Save
output_fn = dir_list['aux_bigmem']+'all_preds.zarr'
utility_save(preds,output_fn,save_kwargs={'zarr_format':2})

/glade/derecho/scratch/schwarzwald/aux_data/all_preds.zarr saved!
